# Model Selection and Comparison — Making the Call Like a Professional

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb14_model_selection_protocol.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Set up a **selection protocol** that evaluates the **full lineage of reference models from nb01-nb13** — 7 candidates for the classification case, 8 for regression — on identical cross-validation folds with a declared primary metric, before any results are seen.
2. Compute the **95% confidence interval** on the champion's CV scores using Student's *t* (the same statistic you learned in nb08); apply the **CI-overlap rule** to decide whether the top model has earned the right to displace the simpler runner-up.
3. Justify the champion pick in **stakeholder language** — what was compared, what was picked, and why.
4. Open the **locked test set exactly once per case** and pronounce an INSIDE / ABOVE / BELOW verdict against the champion's CV confidence interval.
5. Internalize the **singleness rule**: *two demo cases × one ceremony each = one ceremony per real-world business analysis*. Whenever you face a real business problem in the future, you open its test set ONCE — not multiple times.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — one champion-pick exercise per case. Complete both before submitting your notebook.

---

## 💼 Why This Matters

Today is the payoff. Thirteen notebooks of locking discipline (test set sealed, all evaluation via cross-validation on the training set, every new model benchmarked against the Week-2 linear reference) converge into a single structured ceremony: **declare the protocol, compare candidates, pick a champion, open the locked envelope.**

Imagine you are presenting to **the State Health Department's review board** that has to approve MedScreen's breast-cancer screening pipeline, or to **HomeValue Analytics' deployment council** that has to sign off on the property-value prediction tool. Both rooms want the same three things from you:

1. A **defensible champion selection** — not "I tried a bunch of models and this one looked best", but *"every reference model introduced from nb01 onward was declared as a candidate (dummy floor, Week-2 linear, sparse linear, Ridge, single tree, random forest, default GBM, tuned GBM), evaluated under identical 5-fold CV folds with ROC-AUC (classification) or R² (regression) as the primary metric, and the champion was picked because its 95% confidence interval sat above the runner-up's by a clear margin — or, if the intervals overlapped, because it was the simpler model."*
2. A **95% confidence interval** on the champion's training-set performance — the headline number anyone in the room can quote.
3. A **single test-set evaluation** — the one and only authorized opening of the locked envelope — that confirms (or contradicts) what cross-validation predicted.

The third bullet is the part this notebook makes visible. Before today every cell that touched the test set was off-limits. **In §6 the test set opens — once, for each of the two business cases — and never again in the course.** nb15 onward goes back to cross-validation only. The test set is for the final headline number you bring to the stakeholder meeting, not for iteration.

> **A question that often comes up here:** *"why is opening the test set such a big deal?"* Because every time you open the test set and let what you see influence what you do next, you have leaked the test set into model selection — which means the test set can no longer give you an unbiased estimate of how the model will perform on new data. The discipline is not statistical pedantry; it is the only mechanism by which the test-set point estimate has any meaning at all. nb14's ceremony exists to drive that point home in the most visible way possible: open the envelope, write the number down, close the envelope, never reopen.

### The singleness rule (read this carefully)

This notebook walks **two ceremonies** — one for classification, one for regression — because there are two demo business cases. **Whenever you analyze a single real business problem in the future, you open its test set ONCE, not twice.** Two ceremonies in this notebook is a teaching demo, not a precedent: in any single real engagement, the test set's job is to give you one honest verdict, then go back into its envelope.

---

## 1. Setup — Imports, References, Helpers

The setup cell does the same five jobs as nb12 / nb13 plus one new utility: `compare_models_comprehensive()`, the comparison harness that takes a dict of candidate models and returns a sorted DataFrame of CV means with Student's *t* 95% CIs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                              r2_score, mean_squared_error, mean_absolute_error)
import time, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'
GREEN     = '#2ca02c'
RED       = '#d62728'

# --- Week-2 references ---
reference_clf = Pipeline([('scaler', StandardScaler()),
                          ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))])
reference_reg = Pipeline([('scaler', StandardScaler()),
                          ('reg',    LinearRegression())])

# --- Helper: CV-CI dot plot from previous notebooks ---
def plot_cv_ci(scores_dict, metric_name, title, ax, color=CLF_COLOR, k=5,
               highlight=None, highlight_color=GREEN):
    """Dot plot with 95% CIs; optionally highlight one model."""
    t_crit = stats.t.ppf(0.975, df=k - 1)
    rows = []
    for name, scores in scores_dict.items():
        m = float(np.mean(scores)); sd = float(np.std(scores, ddof=1))
        rows.append({'name': name, 'mean': m, 'half_w': t_crit * sd / np.sqrt(k)})
    df = pd.DataFrame(rows).sort_values('mean')
    colors = [highlight_color if (highlight is not None and r['name']==highlight) else color
              for _, r in df.iterrows()]
    for i, (_, r) in enumerate(df.iterrows()):
        ax.errorbar(r['mean'], i, xerr=r['half_w'], fmt='o', capsize=6, linewidth=2,
                    color=colors[i], markersize=10)
        ax.text(r['mean'] + r['half_w'] + (r['half_w']*0.2 if r['half_w']>0 else 0.001),
                i, f"{r['mean']:.4f}", va='center', fontsize=9)
    ax.set_yticks(range(len(df))); ax.set_yticklabels(df['name'])
    ax.set_xlabel(f'5-fold CV {metric_name} (mean ± 95% CI)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# --- Comparison harness ---
def compare_models_comprehensive(models_dict, X, y, cv, scoring, primary_metric):
    """Cross-validate each model on the same folds; return DataFrame with means, SDs, fit times."""
    rows = []
    for name, model in models_dict.items():
        t0 = time.time()
        cvr = cross_validate(model, X, y, cv=cv, scoring=scoring,
                             return_train_score=False, n_jobs=-1)
        fit_time = time.time() - t0
        row = {'Model': name, 'fit_time_s': fit_time}
        for m in scoring:
            row[f'{m}_mean'] = cvr[f'test_{m}'].mean()
            row[f'{m}_sd']   = cvr[f'test_{m}'].std(ddof=1)
            row[f'{m}_folds'] = cvr[f'test_{m}']
        rows.append(row)
    df = pd.DataFrame(rows)
    return df.sort_values(f'{primary_metric}_mean', ascending=False).reset_index(drop=True)

# --- Verdict helper for the ceremony ---
def verdict(test_score, cv_mean, cv_sd, k=5):
    """Compare a test-set point estimate to a 95% CV CI; return INSIDE/ABOVE/BELOW + message."""
    t_crit = stats.t.ppf(0.975, df=k - 1)
    half_w = t_crit * cv_sd / np.sqrt(k)
    lo, hi = cv_mean - half_w, cv_mean + half_w
    if test_score < lo:
        return 'BELOW', f'Test ({test_score:.4f}) is BELOW the CV CI ({lo:.4f}, {hi:.4f}) — model overfit the training data.'
    if test_score > hi:
        return 'ABOVE', f'Test ({test_score:.4f}) is ABOVE the CV CI ({lo:.4f}, {hi:.4f}) — pleasant surprise; investigate why CV underestimated.'
    return 'INSIDE', f'Test ({test_score:.4f}) is INSIDE the CV CI ({lo:.4f}, {hi:.4f}) — CV estimate held; ship the model.'

print("✓ Setup, references, helpers, harness, verdict function — all loaded")


---

## 2. Load Both Datasets — Three Holdout Splits, Two Locked Envelopes

Same 60/20/20 splits as nb11–nb13 with the same `random_state=RANDOM_SEED`. The CV scores you compute here will match (within fold-level fluctuation) the CV scores from previous notebooks because the splits are deterministic. The test envelopes have stayed sealed for three notebooks straight; today is the day they open.

In [ ]:
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target

# 60/20/20 split — identical partition to nb11 / nb12 / nb13 under RANDOM_SEED.
X_clf_temp, X_test_clf, y_clf_temp, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_train_clf, X_val_clf, y_train_clf, y_val_clf = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target

X_reg_temp, X_test_reg, y_reg_temp, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f'Classification: train {len(X_train_clf):>5} | val {len(X_val_clf):>4} | test {len(X_test_clf):>4} (LOCKED until §7.1)')
print(f'Regression:     train {len(X_train_reg):>5} | val {len(X_val_reg):>4} | test {len(X_test_reg):>4} (LOCKED until §7.2)')


---

## 3. The Model Selection Problem

Picking a model is harder than it sounds. The naïve approach — *"just compare the numbers and pick the highest one"* — has six classic failure modes. The right approach has six corresponding fixes:

| What goes wrong | What we do instead |
|---|---|
| Different CV splits per model (apples vs oranges) | **Same CV folds** for every candidate (apples vs apples) |
| Different metric per model ("metric shopping") | **One declared primary metric**, supporting metrics for context only |
| Pick the model by test-set score | Pick by cross-validation; **test set is for the final verdict, not selection** |
| Try new candidates until something wins | **Lock the candidate roster before fitting any model** |
| Ignore fit time | Track fit time as a tie-breaker when CV scores are close |
| No written rationale | Write a 3-sentence explanation in plain stakeholder language |

The protocol below operationalizes the right column. **Section 4** declares the candidate roster; **Section 5** evaluates everything on identical CV folds and visualizes the result; the **PAUSE-AND-DO exercises** ask you to pick a champion using the CI-overlap rule and explain it in stakeholder language; **Section 6** opens the locked test set.

---

## 4. Define the Candidate Roster — Every Reference Since nb01

The candidate roster is **declared before any results are seen**. This rules out "model shopping" — the failure mode where you keep trying new models until one happens to beat the rest. The discipline goes further today: **every reference model introduced from nb01 onward is on the roster, and each one carries the same hyperparameters its source notebook actually shipped**. nb14 is the chain's audit, not a re-tuning step.

Seven candidates per case for classification, eight for regression — the extra reg slot is the Ridge–Lasso pair from nb05.

| # | Family | Source notebook | Classification | Regression |
|---|---|---|---|---|
| 1 | **Dummy floor** (random-guess / mean) | nb03 (reg), nb06 (clf) | `DummyClassifier('most_frequent')` | `DummyRegressor('mean')` |
| 2 | **Week-2 reference** (linear) | nb06/nb08/nb09 (clf), nb03/nb08/nb09 (reg) | `LogReg(C=1.0)` | OLS (`LinearRegression`) |
| 3 | **Ridge regularization** (L2) | nb05 / nb08 | — *(L2 is already the Week-2 default for LogReg)* | `Ridge(alpha=1.0)` |
| 4 | **Sparse linear** (L1) | nb05 (reg), nb09 (clf) | `LogReg L1(C=0.1)` | `Lasso(alpha=0.01)` |
| 5 | **Single tree** | nb11 §6 | `DT(depth=3)` | `DT(depth=5)` |
| 6 | **Bagged ensemble** | nb12 §5 / §9 ship | `RF(n=50, max_features='sqrt', max_depth=3)` | `RF(n=50, max_features=0.5)` |
| 7 | **Boosted ensemble** (default) | nb13 §4 | `GBM(default)` | `GBM(default)` |
| 8 | **Boosted ensemble** (tuned) | nb13 §6 ship | `GBM(lr=0.2, n=200, max_features=0.5, max_depth=3)` | `GBM(lr=0.2, n=200, max_features=0.5, max_depth=3)` |

Three notes on the roster:

- The **dummy floor** is candidate #1 because every sophisticated candidate must clear it by a wide margin to justify its compute cost — `DummyClassifier('most_frequent')` is exactly the random-guess line (ROC-AUC = 0.500) and `DummyRegressor('mean')` is the predict-the-mean baseline (R² ≈ 0.000). Seeing the lift over the floor on the dot plot is a quick sanity check that the more elaborate models are buying real signal, not artifacts.
- The **Week-2 reference** is candidate #2 because it is the linear floor every later model has to clear by a *CI-clear* margin to earn displacement. The dummy is the *absolute* floor; the Week-2 reference is the *useful* floor.
- The classification roster has no separate "Ridge logistic" entry because `LogReg(C=1.0)` is already an L2-regularized linear classifier — the Week-2 reference IS Ridge logistic with the default C. The regression roster keeps Ridge and Lasso as separate entries because nb05 taught both as distinct regularization strategies.

**Hyperparameter provenance.** Rows 6, 7, and 8 use the **exact configurations nb12 and nb13 shipped**, not generic defaults:

- Row 6 — the bagged ensemble — uses nb12 §5's 3D joint-grid picks. **Classification**: `RandomForestClassifier(n=50, max_features='sqrt', max_depth=3)` — nb11's depth survives joint tuning under CI-overlap + parsimony; sklearn's clf default `max_features='sqrt'` paired with the smallest tree count. **Regression**: `RandomForestRegressor(n=50, max_features=0.5)` (max_depth defaults to None) — the §5 3D grid revealed depth-3 and depth-5 cells sit CI-clear below depth-None on California Housing; the depth-None tied group's largest-mean-smallest-CI cell at n=50 lives at `max_features=0.5`.
- Row 7 — the default GBM — uses sklearn's defaults exactly as nb13 §4 introduced them (`learning_rate=0.1, n_estimators=100, max_depth=3, max_features=None`).
- Row 8 — the tuned GBM — uses nb13 §6's joint 4D `(learning_rate × n_estimators × max_features × max_depth)` grid winners. **Both cases land on the same configuration**: `(lr=0.2, n_estimators=200, max_features=0.5, max_depth=3)` — sklearn's default GBM depth + nb12's regression `max_features` pick + the high-lr × many-trees corner of the §5 diagonal sweet spot.

In [ ]:
clf_models = {
    'Dummy (most_frequent)':                          DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED),
    'Week-2 ref: LogReg(C=1.0)':                      reference_clf,
    'LogReg L1 (C=0.1)':                              Pipeline([('scaler', StandardScaler()),
                                                                ('clf', LogisticRegression(penalty='l1', C=0.1, solver='liblinear',
                                                                                           random_state=RANDOM_SEED, max_iter=5000))]),
    'Decision Tree (depth=3) [nb11 ship]':            DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
    'Random Forest (n=50, sqrt, d=3) [nb12 ship]':    RandomForestClassifier(n_estimators=50, max_features='sqrt', max_depth=3,
                                                                              random_state=RANDOM_SEED, n_jobs=-1),
    'GBM default [nb13]':                             GradientBoostingClassifier(random_state=RANDOM_SEED),
    'GBM tuned (lr=0.2, n=200, mf=0.5, d=3) [nb13 ship]': GradientBoostingClassifier(n_estimators=200, learning_rate=0.2,
                                                                                      max_features=0.5, max_depth=3,
                                                                                      random_state=RANDOM_SEED),
}

reg_models = {
    'Dummy (mean)':                                   DummyRegressor(strategy='mean'),
    'Week-2 ref: OLS':                                reference_reg,
    'Ridge (alpha=1.0)':                              Pipeline([('scaler', StandardScaler()),
                                                                ('reg', Ridge(alpha=1.0, random_state=RANDOM_SEED))]),
    'Lasso (alpha=0.01)':                             Pipeline([('scaler', StandardScaler()),
                                                                ('reg', Lasso(alpha=0.01, random_state=RANDOM_SEED, max_iter=5000))]),
    'Decision Tree (depth=5) [nb11 ship]':            DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED),
    'Random Forest (n=50, mf=0.5) [nb12 ship]':       RandomForestRegressor(n_estimators=50, max_features=0.5,
                                                                             random_state=RANDOM_SEED, n_jobs=-1),
    'GBM default [nb13]':                             GradientBoostingRegressor(random_state=RANDOM_SEED),
    'GBM tuned (lr=0.2, n=200, mf=0.5, d=3) [nb13 ship]': GradientBoostingRegressor(n_estimators=200, learning_rate=0.2,
                                                                                     max_features=0.5, max_depth=3,
                                                                                     random_state=RANDOM_SEED),
}

print(f"✓ {len(clf_models)} classification candidates, {len(reg_models)} regression candidates declared")
print("✓ Random Forest and tuned GBM hyperparameters match the actual ship picks from nb12 §5 (3D joint grid) and nb13 §6 (4D joint grid)")
print("✓ Roster locked — no candidates added/removed after this cell")

---

## 5. Multi-Metric Reporting — CV Means with 95% CIs

For each case, evaluate every candidate under the **same** 5-fold CV folds. Track the primary metric plus two supporting metrics:

- **Classification:** ROC-AUC (primary), accuracy, F1 (supporting)
- **Regression:** R² (primary), neg-RMSE, neg-MAE (supporting)

The CV-CI dot plot for the primary metric is the **central visual** of the selection ceremony. The companion table reports all metrics + fit times. The runner-up question — *"is the top model's CI clearly above the runner-up's?"* — is what the next section adjudicates.

In [ ]:
clf_scoring = ['roc_auc', 'accuracy', 'f1']
reg_scoring = ['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error']

clf_results = compare_models_comprehensive(clf_models, X_train_clf, y_train_clf,
                                            cv=cv_clf, scoring=clf_scoring, primary_metric='roc_auc')
reg_results = compare_models_comprehensive(reg_models, X_train_reg, y_train_reg,
                                            cv=cv_reg, scoring=reg_scoring, primary_metric='r2')

print("=== CLASSIFICATION (ranked by CV ROC-AUC) ===")
print(clf_results[['Model', 'roc_auc_mean', 'roc_auc_sd', 'accuracy_mean', 'f1_mean', 'fit_time_s']]
      .to_string(index=False))
print()
print("=== REGRESSION (ranked by CV R²) ===")
print(reg_results[['Model', 'r2_mean', 'r2_sd',
                   'neg_root_mean_squared_error_mean', 'neg_mean_absolute_error_mean',
                   'fit_time_s']].to_string(index=False))

# Build dicts for plot_cv_ci
clf_dict = {row['Model']: row['roc_auc_folds'] for _, row in clf_results.iterrows()}
reg_dict = {row['Model']: row['r2_folds']      for _, row in reg_results.iterrows()}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_cv_ci(clf_dict, 'ROC-AUC', 'Classification — 7 candidates ranked (full nb01-nb13 lineage)', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_dict, 'R²',      'Regression — 8 candidates ranked (full nb01-nb13 lineage)',     axes[1], color=REG_COLOR)
fig.suptitle('Every reference model since nb01 on one plot, ranked by CV mean of the primary metric',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The dot plots are the central artifact of the selection ceremony. Each candidate's mean is the dot; the horizontal bar is its **95% confidence interval** (the range we'd expect to see in repeated cross-validation runs). The CI-overlap rule becomes a visual question — overlapping intervals mean statistical tie (simpler model wins); non-overlapping intervals mean the top model has earned displacement.

**The dummy floor anchors both panels.** `DummyClassifier('most_frequent')` lands at ROC-AUC = **0.500** — exactly the random-guess line — and `DummyRegressor('mean')` lands at R² ≈ **0.000** — the predict-the-mean baseline from nb03. Every sophisticated candidate clears the floor by a wide margin on both business cases, which is the quickest possible sanity check that the work is buying real signal rather than memorizing artifacts.

**The MedScreen verdict (classification).** Wisconsin breast cancer is a small dataset (\~341 training patients) and mostly linearly separable in standardized cell-nucleus space. The top candidates' confidence intervals overlap heavily: the Week-2 reference `LogReg(C=1.0)` leads at CV ROC-AUC ≈ **0.994** with a tight 95% CI \~**[0.986, 1.001]**; `LogReg L1` (\~0.990), `GBM tuned (lr=0.2, n=200, mf=0.5, d=3)` from nb13 §6 (\~0.986), the nb12 §5 ship pick `Random Forest (n=50, mf='sqrt', d=3)` (\~0.981), and default GBM (\~0.981) all sit inside that interval; only the depth-3 decision tree (CV ≈ 0.922) falls noticeably below. **By the CI-overlap parsimony rule, MedScreen's champion is the Week-2 `LogisticRegression(C=1.0)`** — the ensembles competed admirably but did not earn the right to displace the simplest linear model from nb06. This matches **both nb12 §9 and nb13 §7**'s verdict on the classification spine, which is exactly the kind of corroboration nb14's ceremony is designed to expose. For the State Health Department's review board, this is good news: the recommended model is the one with the most transparent decision boundary, which is what an oncologist can explain to a patient.

**The HomeValue verdict (regression).** California Housing is a 36× larger dataset and the picture is more spread out. The three linear candidates (OLS, Ridge α=1.0, Lasso α=0.01) cluster at CV R² ≈ **0.586** — Ridge and OLS are visually indistinguishable, empirically confirming nb09's grid-search finding that no Ridge α displaces OLS on this data. The single tree lifts to **0.613**; default GBM to **0.784**; and the top two candidates land in a near-tie: **tuned GBM `(lr=0.2, n=200, mf=0.5, d=3)` at CV R² ≈ 0.812 with 95% CI [0.798, 0.825]**, and **Random Forest `(n=50, mf=0.5)` at CV R² ≈ 0.804 with 95% CI [0.794, 0.814]**. Those two CIs **overlap** on roughly **[0.798, 0.814]** — by the CI-overlap rule from nb08, the tuned GBM and the random forest are **statistically tied**, and the **simpler model class wins by parsimony**. **HomeValue's champion is the `RandomForestRegressor(n_estimators=50, max_features=0.5)` from nb12** — same pick nb12 §9 and nb13 §7 made under the same rule. The tuned GBM is on the candidate roster as the runner-up; if a future re-run pulled the CIs apart, GBM would be next in line.

> **A question that often comes up here:** *"if the tuned GBM has the higher mean, why does the random forest win on regression?"* Because the **CI-overlap rule treats overlapping intervals as a statistical tie**, and a tie defaults to the simpler model. Boosting is sequential, harder to tune, more prone to overfitting under retraining, and harder to explain to a non-technical audience than bagging — three reasons a deployment council would prefer a forest at equal predictive performance. The 0.8 R² mean lift of the tuned GBM over the random forest is real but not large enough (under fold-to-fold uncertainty on a 12,384-row training set) to make the GBM's added complexity worthwhile. This is the same logic nb09 used to keep OLS over Ridge: when statistical evidence ties, the simpler model wins.

> **A question that often comes up here:** *"why bother including Ridge and the default GBM if neither becomes the champion?"* Because the runner-up rows are doing real teaching work. Ridge α=1.0 sitting on top of OLS confirms that nb09's empirical conclusion (no Ridge α dominates OLS on this data) generalizes from the random-search grid to today's ceremony. Default GBM landing well behind both the random forest and the tuned GBM quantifies what nb13's tuning bought — roughly 3 R² points and a CI-clear gap over the default. The full roster is a complete record of the course's modeling chain; deleting candidates because they are not the champion would erase the evidence behind the verdict.

**Key takeaway:** The expanded roster anchored to nb12 / nb13's actual ship picks produces the same two champions both source notebooks already named under the same rule. **MedScreen ships `LogisticRegression(C=1.0)`** (ensembles tied with the Week-2 reference; parsimony picks the simpler linear model). **HomeValue ships `RandomForestRegressor(n_estimators=50, max_features=0.5)`** (tuned GBM has the higher mean, but the CIs overlap and parsimony picks the bagged ensemble over the boosted one). The PAUSE-AND-DO exercises below ask you to apply the CI-overlap rule yourself, set the champion variables for the §6 ceremony, and write a 3-sentence stakeholder explanation for each case.

---

## 📝 PAUSE-AND-DO Exercise 1 (Classification, 7 minutes) — Pick the Champion and Explain Why

The §5 dot plot just showed you seven candidates ranked by CV ROC-AUC. Your job: apply the **CI-overlap rule** to pick the classification champion, set the variables §6's ceremony will need, then explain the pick in stakeholder language for the State Health Department's review board.

**Steps:**

1. From `clf_results`, pull the **top row by mean** (the champion candidate) and the **runner-up** (second row).
2. Compute the **95% CI half-width** for both: `t_crit × sd / √k` where `t_crit = scipy.stats.t.ppf(0.975, df=4)` and `k = 5`.
3. Check whether the two 95% CIs overlap.
   - **CIs overlap** → statistical tie → the **simpler model wins** by parsimony (use `fit_time_s` as the tie-breaker when both are linear-ish).
   - **CIs do NOT overlap** → the top model wins outright (CI-clear margin).
4. Set `champ_clf_name`, `champ_clf_mean`, `champ_clf_sd` to the chosen champion's `Model`, `roc_auc_mean`, and `roc_auc_sd`. **The §6 ceremony cell will not run without these variables.**
5. Below the code cell, fill in the markdown template with a **3-sentence explanation** the State Health Department's review board could read aloud.

---

> 💡 **Gemini Prompt:** "From `clf_results` (a pandas DataFrame already ranked by `roc_auc_mean` in descending order), pull `iloc[0]` (champion candidate) and `iloc[1]` (runner-up). Compute the 95% CI half-width for each using `t_crit = scipy.stats.t.ppf(0.975, df=4)` and `k = 5`. Check whether the two intervals overlap. If they do, pick the simpler model (use the row with lower `fit_time_s` as the tie-breaker) — if not, keep the top row as champion. Set `champ_clf_name`, `champ_clf_mean`, `champ_clf_sd` accordingly. Print the champion's name, its 95% CI bounds, and a one-line verdict explaining whether the rule kept the top row or swapped to the runner-up."
>
> **After running, verify:**
> - [ ] `champ_clf_name`, `champ_clf_mean`, `champ_clf_sd` are all defined
> - [ ] One-line CI-overlap verdict is printed
> - [ ] Champion's 95% CI is printed (e.g., `[0.986, 1.001]`)

In [ ]:
# YOUR SOLUTION CODE HERE
# Identify the classification champion using the CI-overlap rule.
# Required output variables (used by §6 ceremony):
#   champ_clf_name : str
#   champ_clf_mean : float
#   champ_clf_sd   : float


## 📝 PAUSE-AND-DO Exercise 2 (Regression, 7 minutes) — Pick the Champion and Explain Why

Same template, the regression case. The §5 dot plot showed you eight candidates ranked by CV R². Apply the **CI-overlap rule** to pick the regression champion, set the variables §6's ceremony will need, then explain the pick for HomeValue Analytics' deployment council.

**Steps:**

1. From `reg_results`, pull the **top row** (champion candidate) and the **runner-up** (second row).
2. Compute the 95% CI half-width for both: `t_crit × sd / √k` where `t_crit = scipy.stats.t.ppf(0.975, df=4)` and `k = 5`.
3. Check whether the two 95% CIs overlap.
   - **CIs overlap** → statistical tie → **simpler model wins** by parsimony.
   - **CIs do NOT overlap** → the top model wins outright.
4. Set `champ_reg_name`, `champ_reg_mean`, `champ_reg_sd`. **The §6 ceremony cell will not run without these variables.**
5. Below the code cell, fill in the markdown template. Convert R² to a USD-RMSE intuition when justifying — *"the champion's CV R² is X, which translates to roughly USD Y of typical prediction error per property"*. Tie the operational revisit trigger to housing-market dynamics (median income shifts, new construction permits, etc.).

---

> 💡 **Gemini Prompt:** "From `reg_results` (a pandas DataFrame already ranked by `r2_mean` in descending order), pull `iloc[0]` (champion candidate) and `iloc[1]` (runner-up). Compute the 95% CI half-width for each using `t_crit = scipy.stats.t.ppf(0.975, df=4)` and `k = 5`. Check whether the two intervals overlap. If overlap → pick the simpler model (lower `fit_time_s` wins as tie-breaker). If no overlap → keep the top row. Set `champ_reg_name`, `champ_reg_mean`, `champ_reg_sd`. Print the champion name, its 95% CI bounds, the equivalent CV-RMSE in USD (multiply the RMSE column by 100,000), and a one-line CI-overlap verdict."
>
> **After running, verify:**
> - [ ] `champ_reg_name`, `champ_reg_mean`, `champ_reg_sd` are all defined
> - [ ] One-line CI-overlap verdict is printed
> - [ ] Champion's 95% CI is printed in R² units AND CV-RMSE is printed in USD

In [ ]:
# YOUR SOLUTION CODE HERE
# Identify the regression champion using the CI-overlap rule.
# Required output variables (used by §6 ceremony):
#   champ_reg_name : str
#   champ_reg_mean : float
#   champ_reg_sd   : float


## 6. Opening the Locked Test Set — The Ceremony

The candidate roster is locked. The CV ranking is published. The champion is picked. **Now — and only now — the locked test set opens.** Once per case, never again.

The ceremony has the same structure on both spines:

1. Refit the champion on **all** training data (no holdout — we want every drop of training signal in the deployed model).
2. Predict on the locked test set.
3. Compute the test-set point estimate of the primary metric.
4. Compare the test-set point to the **CV 95% confidence interval** from §5 — render the verdict as **INSIDE / ABOVE / BELOW**.

Each verdict has a meaning:

- **INSIDE** → the test point lands within the CV 95% CI. Cross-validation predicted what we observed. Ship the model.
- **ABOVE** → the test point lands above the CV CI. A pleasant surprise; the model performs better on the test set than CV suggested. Worth investigating *why* CV underestimated (could be a lucky split; could be a real distributional shift in the test partition).
- **BELOW** → the test point lands below the CV CI. The model overfit the training data. Something leaked from the training/validation pipeline; do not ship without diagnosing the failure.

### 6.1 Classification Ceremony — Wisconsin Breast Cancer

In [ ]:
# CEREMONY — CLASSIFICATION
# X_test_clf opens here, exactly once. After this cell, it goes back into the envelope.

# Fallback: if Exercise 1 left the champion variables undefined (YOUR SOLUTION
# CODE HERE was not filled in), auto-pick the champion via the CI-overlap rule
# so the ceremony still runs end-to-end.
if 'champ_clf_name' not in globals() or 'champ_clf_mean' not in globals() or 'champ_clf_sd' not in globals():
    _top    = clf_results.iloc[0]
    _runner = clf_results.iloc[1]
    _t      = stats.t.ppf(0.975, df=4)
    _h_top  = _t * _top['roc_auc_sd']    / np.sqrt(5)
    _h_run  = _t * _runner['roc_auc_sd'] / np.sqrt(5)
    _overlap = (max(_top['roc_auc_mean'] - _h_top, _runner['roc_auc_mean'] - _h_run)
                <= min(_top['roc_auc_mean'] + _h_top, _runner['roc_auc_mean'] + _h_run))
    _pick = _runner if (_overlap and _runner['fit_time_s'] < _top['fit_time_s']) else _top
    champ_clf_name = _pick['Model']
    champ_clf_mean = _pick['roc_auc_mean']
    champ_clf_sd   = _pick['roc_auc_sd']
    print(f"⚠️  Exercise 1 left empty — auto-picked champion via CI-overlap rule: {champ_clf_name}")
    print()

champion_clf = clf_models[champ_clf_name]
champion_clf.fit(X_train_clf, y_train_clf)

# Get probability predictions for ROC-AUC and class predictions for accuracy/F1
y_proba_test_clf = champion_clf.predict_proba(X_test_clf)[:, 1]
y_pred_test_clf  = champion_clf.predict(X_test_clf)

test_auc      = roc_auc_score(y_test_clf, y_proba_test_clf)
test_accuracy = accuracy_score(y_test_clf, y_pred_test_clf)
test_f1       = f1_score(y_test_clf, y_pred_test_clf)

verdict_label_clf, verdict_msg_clf = verdict(test_auc, champ_clf_mean, champ_clf_sd)

print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'CEREMONY — CLASSIFICATION (Wisconsin Breast Cancer)')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Champion:           {champ_clf_name}')
print(f'CV ROC-AUC mean:    {champ_clf_mean:.4f}  (SD = {champ_clf_sd:.4f})')
print(f'Test ROC-AUC:       {test_auc:.4f}')
print(f'Test accuracy:      {test_accuracy:.4f}')
print(f'Test F1:            {test_f1:.4f}')
print(f'')
print(f'VERDICT: {verdict_label_clf}')
print(f'{verdict_msg_clf}')

# THE money plot for classification
t_crit = stats.t.ppf(0.975, df=4)
half_w = t_crit * champ_clf_sd / np.sqrt(5)
fig, ax = plt.subplots(figsize=(11, 4))
verdict_color = {'INSIDE': GREEN, 'ABOVE': CLF_COLOR, 'BELOW': RED}[verdict_label_clf]
ax.errorbar([champ_clf_mean], [0], xerr=[half_w], fmt='o', capsize=10,
            color=GREY, markersize=12, linewidth=3, label=f'CV mean ± 95% CI: {champ_clf_mean:.4f} ± {half_w:.4f}')
ax.scatter([test_auc], [0], color=verdict_color, marker='^', s=300, zorder=5,
           label=f'Test point: {test_auc:.4f}  →  {verdict_label_clf}')
ax.axvline(champ_clf_mean - half_w, color=GREY, linestyle=':', alpha=0.5)
ax.axvline(champ_clf_mean + half_w, color=GREY, linestyle=':', alpha=0.5)
ax.set_yticks([]); ax.set_xlabel('ROC-AUC')
ax.set_title(f'Classification ceremony — test ROC-AUC vs CV CI of the champion ({champ_clf_name})',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


**Reading the output:**

The triangle is the test-set point estimate; the dot is the CV mean; the dotted vertical lines mark the 95% CI bounds. If the triangle is between the dotted lines (INSIDE), the CV estimate held — ship the model. If the triangle is above the upper dotted line (ABOVE), the test set was kinder to the model than CV expected — investigate but do not panic. If the triangle is below the lower dotted line (BELOW), the model overfit the training data and the test-set number is the honest one.

Most of the time the verdict is INSIDE because CV-with-CI is well-calibrated under i.i.d. assumptions. ABOVE happens occasionally on small datasets (the test fold landed easy). BELOW is the dangerous case — it usually signals a leakage problem in the training pipeline that CV did not catch.

**The classification test set is now closed. It will not reopen in this course.**

---

### 6.2 Regression Ceremony — California Housing

Same protocol on the regression side. Champion is the tuned GBM picked in §5 / PAUSE-AND-DO 2; the locked test set `X_test_reg` opens for exactly one prediction pass.

In [ ]:
# CEREMONY — REGRESSION
# X_test_reg opens here, exactly once. After this cell, it goes back into the envelope.

# Fallback: if Exercise 2 left the champion variables undefined, auto-pick via
# the same CI-overlap rule so the ceremony still runs end-to-end.
if 'champ_reg_name' not in globals() or 'champ_reg_mean' not in globals() or 'champ_reg_sd' not in globals():
    _top    = reg_results.iloc[0]
    _runner = reg_results.iloc[1]
    _t      = stats.t.ppf(0.975, df=4)
    _h_top  = _t * _top['r2_sd']    / np.sqrt(5)
    _h_run  = _t * _runner['r2_sd'] / np.sqrt(5)
    _overlap = (max(_top['r2_mean'] - _h_top, _runner['r2_mean'] - _h_run)
                <= min(_top['r2_mean'] + _h_top, _runner['r2_mean'] + _h_run))
    _pick = _runner if (_overlap and _runner['fit_time_s'] < _top['fit_time_s']) else _top
    champ_reg_name = _pick['Model']
    champ_reg_mean = _pick['r2_mean']
    champ_reg_sd   = _pick['r2_sd']
    print(f"⚠️  Exercise 2 left empty — auto-picked champion via CI-overlap rule: {champ_reg_name}")
    print()

champion_reg = reg_models[champ_reg_name]
champion_reg.fit(X_train_reg, y_train_reg)
y_pred_test_reg = champion_reg.predict(X_test_reg)

test_r2   = r2_score(y_test_reg, y_pred_test_reg)
test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_test_reg))
test_mae  = mean_absolute_error(y_test_reg, y_pred_test_reg)

verdict_label_reg, verdict_msg_reg = verdict(test_r2, champ_reg_mean, champ_reg_sd)

print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'CEREMONY — REGRESSION (California Housing)')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Champion:           {champ_reg_name}')
print(f'CV R² mean:         {champ_reg_mean:.4f}  (SD = {champ_reg_sd:.4f})')
print(f'Test R²:            {test_r2:.4f}')
print(f'Test RMSE:          {test_rmse:.4f}  (USD {test_rmse * 100_000:,.0f})')
print(f'Test MAE:           {test_mae:.4f}   (USD {test_mae  * 100_000:,.0f})')
print(f'')
print(f'VERDICT: {verdict_label_reg}')
print(f'{verdict_msg_reg}')

# Money plot for regression
t_crit = stats.t.ppf(0.975, df=4)
half_w_r = t_crit * champ_reg_sd / np.sqrt(5)
fig, ax = plt.subplots(figsize=(11, 4))
verdict_color = {'INSIDE': GREEN, 'ABOVE': REG_COLOR, 'BELOW': RED}[verdict_label_reg]
ax.errorbar([champ_reg_mean], [0], xerr=[half_w_r], fmt='o', capsize=10,
            color=GREY, markersize=12, linewidth=3,
            label=f'CV mean ± 95% CI: {champ_reg_mean:.4f} ± {half_w_r:.4f}')
ax.scatter([test_r2], [0], color=verdict_color, marker='^', s=300, zorder=5,
           label=f'Test point: {test_r2:.4f}  →  {verdict_label_reg}')
ax.axvline(champ_reg_mean - half_w_r, color=GREY, linestyle=':', alpha=0.5)
ax.axvline(champ_reg_mean + half_w_r, color=GREY, linestyle=':', alpha=0.5)
ax.set_yticks([]); ax.set_xlabel('R²')
ax.set_title(f'Regression ceremony — test R² vs CV CI of the champion ({champ_reg_name})',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


**Reading the output:**

Same money plot, regression version. Triangle = test point; dot = CV mean; dotted lines = 95% CI bounds. The verdict carries the same three options. For California Housing with the tuned GBM, the verdict is usually INSIDE — the CV-with-CI estimate is well-calibrated and the held-out RMSE typically lands within USD 5K of the CV-RMSE.

Translating R² to USD-RMSE is the version stakeholders will quote. *"R² = 0.83"* is abstract; *"the model is typically off by USD 41,000 on a USD 200,000 home"* is a number HomeValue's deployment council can defend or contest in concrete terms.

**The regression test set is now closed. It will not reopen in this course.**

---

### 6.3 The Singleness Rule — Two Demos, One Ceremony Per Real Business Case

You just watched **two ceremonies** because there are two demo business cases in this notebook. Any single real business problem you will analyze in the future is **one** case — not two. For any single engagement, you open the test set **ONCE**, not twice.

| What this notebook did | What you do on a real business case |
|---|---|
| Loaded breast cancer + California Housing | Loads your one project dataset |
| Ran 7- or 8-candidate selection on each business case | Runs candidate selection on your one case |
| Wrote two stakeholder explanations | Writes one stakeholder explanation |
| Opened `X_test_clf` once + `X_test_reg` once | Opens your test set ONCE |
| Two INSIDE/ABOVE/BELOW verdicts (one per case) | One INSIDE/ABOVE/BELOW verdict |

The discipline is **per business case**, not per notebook. Two ceremonies in this notebook is a teaching demo, not a precedent. **A single test set should never be opened twice in a single business engagement.** If you find yourself wanting to "just check" the test set after seeing a CV result, that is the moment to close the laptop and re-read this section.

> **A question that often comes up here:** *"what if my INSIDE verdict turns into a BELOW verdict on the test set — can I just add another candidate and re-run?"* No. Adding a candidate after seeing the test result is **selection on the test set**, which destroys the test set's role. Three legitimate paths forward: (1) **accept the BELOW verdict** and report the test number honestly — the model is what the test set says it is, and the CV was optimistic; (2) **investigate the BELOW verdict** (usually a leakage in your training pipeline), fix it, re-do CV without touching the test set, and *only then* consider re-opening the test for a second ceremony in a different notebook; or (3) **get more data** and re-do everything from nb09 forward. What you cannot do is keep iterating on the same training/test split until the verdict turns favorable — that is the very pattern this whole course is designed to prevent.

---

## 7. Wrap-Up — Key Takeaways

**What landed today:**

1. **The selection protocol is a structured ceremony, not a spreadsheet exercise.** Every reference model introduced from nb01-nb13 enters the roster (7 classification candidates, 8 regression candidates), with **the exact hyperparameters its source notebook shipped**. Declared in advance, evaluated on identical CV folds, with one declared primary metric and supporting metrics for tie-breaking.
2. **The CV-CI dot plot is the central artifact.** CIs that overlap → simpler model wins. CIs that do not overlap → the top model has earned displacement.
3. **The champion is justified in stakeholder language.** Three sentences: what model are we shipping, what's the CV confidence interval on its performance, and what would trigger a revisit.
4. **The locked test set opens exactly once per case.** Two ceremonies in this notebook is a teaching demo; in any single real business analysis, the test set opens once and only once.
5. **The verdict is INSIDE / ABOVE / BELOW** — not a number to debate, but a categorical pronouncement against the CV CI.

**The two champions, named:**

- **Classification (MedScreen)** — `LogisticRegression(C=1.0)`. CIs of the top four candidates (LogReg, LogReg L1, Random Forest `(n=50, sqrt, d=3)`, tuned GBM `(lr=0.2, n=200, mf=0.5, d=3)`) overlap heavily; parsimony picks the simplest linear baseline. Same verdict nb12 §9 and nb13 §7 reached under the same rule.
- **Regression (HomeValue)** — `RandomForestRegressor(n_estimators=50, max_features=0.5)` — the nb12 §5 3D joint-grid ship pick. Tuned GBM `(lr=0.2, n=200, mf=0.5, d=3)` has the higher CV R² mean (\~0.812) but its CI overlaps the random forest's (\~0.804); parsimony picks the simpler bagged ensemble. Same verdict nb12 §9 and nb13 §7 reached.

**Bridge to nb15 — Interpretation and Error Analysis:**

The champions are now committed. nb15 takes both committed champions forward (LogReg for classification, the random forest for regression) into the interpretation pass: permutation importance, partial dependence plots, and segment-level error analysis. The four-method importance heatmap from nb12 carries forward as the structural reference; partial dependence plots add the *shape* dimension — *"how does the prediction change as the input feature varies, holding everything else fixed?"*

> **A question that often comes up at this point:** *"if the CV-CI dot plot is so good at picking a champion, why does nb15 even exist?"* Because picking a champion is not the same as understanding what the champion learned. nb15's interpretation pass is the layer that translates the model into stakeholder-readable findings: *"this is what the model is paying attention to; this is where it fails; this is the segment-level fairness audit."* The champion is the *who*; nb15 is the *why and how*.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (pick the classification champion + explain) and Exercise 2 (pick the regression champion + explain).
2. **Run All Cells** — `Runtime → Run all` to execute every cell including the two §6 ceremonies.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 14 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both champion-pick code cells set the required variables (`champ_clf_name/_mean/_sd`, `champ_reg_name/_mean/_sd`)
- [ ] Both champion explanations are written in stakeholder language (3 sentences each)
- [ ] Both money plots (test point vs CV CI) render with the verdict color coded
- [ ] You can defend the singleness rule in plain English

### Next Step:

- **Notebook 15** — Interpretation and Error Analysis (Day 15)

---

<center>

**Thank you!**

</center>